# Phase 6 — Real-World Fine-Tuning (Continuous Learning)
**Notebook độc lập** — chạy sau khi đã thu thập DataRealTest từ testPC.py

## Quy trình
```
DataRealTest (ảnh thực từ Pi camera)  →  80/20 split  →  Dataset_v2
        ↓
   Phase 6 Fine-Tuning  (LR siêu nhỏ, chống Catastrophic Forgetting)
        ↓
   best_model_p6.pth  →  export ONNX v2  →  deploy testPC.py
```

## Tham số LR so sánh
| Phase | LR head | LR backbone | Mục đích |
|---|---|---|---|
| Phase 3 (full train) | 2.5e-4 | 8e-6 | Học từ đầu |
| Phase 5 (lighting) | 5e-5 | 2e-6 | Domain adapt |
| **Phase 6 (realworld)** | **3e-5** | **1e-6** | Tinh chỉnh nhẹ |

> DataRealTest là **Hard Negatives** — ảnh thực tế mà model đã từng nhận sai.
> Chúng quý gấp 10 lần ảnh tải từ internet.


In [ ]:
# ============================================================
# CELL 1 - IMPORTS & HARDWARE
# ============================================================
import os, sys, time, copy, json, math, gc, warnings, contextlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR
from torchvision import transforms
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from PIL import Image
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

%pip install tqdm -q

from tqdm import tqdm

warnings.filterwarnings('ignore')

print('\n' + '='*70)
print('  WASTE DETECTION — Phase 6 Real-World Fine-Tuning')
print('='*70)

if torch.cuda.is_available():
    GPU_COUNT = torch.cuda.device_count()
    GPU_MEM   = torch.cuda.get_device_properties(0).total_memory / 1024**3
    DEVICE    = 'cuda'
    torch.backends.cudnn.benchmark = True
    print(f'  CUDA! GPUs: {GPU_COUNT}')
    for i in range(GPU_COUNT):
        print(f'    GPU {i}: {torch.cuda.get_device_name(i)} '
              f'({torch.cuda.get_device_properties(i).total_memory/1024**3:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    GPU_COUNT, GPU_MEM, DEVICE = 1, 0, 'mps'
    print('  Apple MPS')
else:
    GPU_COUNT, GPU_MEM, DEVICE = 0, 0, 'cpu'
    print('  CPU only')

# -- Auto data_dir ------------------------------------------------
if os.path.exists('/kaggle/input'):
    _found = None
    for root, dirs, _ in os.walk('/kaggle/input'):
        if any(d in dirs for d in ['Battery', 'battery', 'Biological', 'biological']):
            _found = root
            break
    _DEFAULT_DATA = _found or '/kaggle/input'
    _DEFAULT_OUT  = '/kaggle/working/outputs'
    PLATFORM = 'Kaggle'
else:
    _candidates = [
        './Model/DataSet/Data', './Model/DataSet',
        './DataSet/Data', './DataSet', './dataset', './data',
        '../Model/DataSet/Data', '../Model/DataSet',
        '../DataSet/Data', '../DataSet', '../dataset', '../data',
        '/Users/capkimkhanh/Downloads/DataSet',
    ]
    _DEFAULT_DATA = None
    for _p in _candidates:
        if os.path.isdir(_p):
            _DEFAULT_DATA = _p
            break
    _DEFAULT_DATA = _DEFAULT_DATA or './Model/DataSet/Data'
    _DEFAULT_OUT  = './outputs'
    PLATFORM = 'Local'

print(f'  Platform: {PLATFORM} | Data: {_DEFAULT_DATA}')

# -- Batch sizes --------------------------------------------------
_SCALE = max(GPU_COUNT, 1)
if DEVICE == 'cuda' and GPU_MEM >= 14:
    _B = {'p1': 24, 'p2': 16, 'p3': 8, 'p4': 8}
elif DEVICE == 'cuda':
    _B = {'p1': 12, 'p2': 8,  'p3': 4, 'p4': 4}
elif DEVICE == 'mps':
    _B = {'p1': 20, 'p2': 10, 'p3': 8, 'p4': 8}
else:
    _B = {'p1': 4,  'p2': 4,  'p3': 2, 'p4': 2}

for ph, b in _B.items():
    print(f'  Batch {ph.upper()}: {b}/GPU -> {b*_SCALE} effective')

_IN_NOTEBOOK = 'ipykernel' in sys.modules
if PLATFORM == 'Local' and _IN_NOTEBOOK and DEVICE != 'cuda':
    _NUM_WORKERS = 0
elif PLATFORM == 'Local' and _IN_NOTEBOOK and DEVICE == 'cuda':
    _NUM_WORKERS = 2
else:
    _NUM_WORKERS = 2
print(f'  Workers: {_NUM_WORKERS} | Notebook: {_IN_NOTEBOOK}')

if DEVICE == 'mps':
    try:
        torch.mps.set_per_process_memory_fraction(0.72)
        print('  MPS memory fraction: 72%')
    except AttributeError:
        pass
print('='*70)


In [ ]:
# ============================================================
# CELL 2 - CONFIG & CLASSES  (v5 - Adaptive Gamma)
# ============================================================

CLASSES = [
    'Battery',
    'Biological',
    'General_Waste',
    'Glass',
    'Metal',
    'Paper_Cardboard',
    'Plastic',
]

CLASS_ALIASES = {
    'General Waste khẩu trang tàn thuốc găng tay': 'General_Waste',
    'General_Waste': 'General_Waste',
    'General Waste': 'General_Waste',
    'Paper+Cardboard gộp chung thành paper': 'Paper_Cardboard',
    'Paper+Cardboard': 'Paper_Cardboard',
    'Paper_Cardboard': 'Paper_Cardboard',
    'Paper': 'Paper_Cardboard',
}

def _norm_name(name):
    return name.strip().lower().replace('-', '_').replace(' ', '_')

CANONICAL_BY_NORM = {_norm_name(c): c for c in CLASSES}
for alias_name, canonical_name in CLASS_ALIASES.items():
    CANONICAL_BY_NORM[_norm_name(alias_name)] = canonical_name

NUM_CLASSES = len(CLASSES)
CLASS2IDX   = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS   = {i: c for c, i in CLASS2IDX.items()}

if DEVICE == 'mps':
    _P2_IMG = 448
else:
    _P2_IMG = 540

CONFIG = {
    'data_dir'             : _DEFAULT_DATA,
    'output_dir'           : _DEFAULT_OUT,
    'img_size'             : 384,
    'num_workers'          : _NUM_WORKERS,
    'device'               : DEVICE,
    'gpu_count'            : max(GPU_COUNT, 1),
    'seed'                 : 42,
    'val_split'            : 0.2,
    'weight_decay'         : 1e-4,
    'ema_decay'            : 0.9995,
    'focal_gamma'          : 2.0,
    'ema_eval_freq'        : 2,
    'sampler_power'        : 0.7,
    'alpha_power'          : 0.7,
    'min_samples_per_class': 20,

    # ── Adaptive Gamma Correction ─────────────────────────────────────────
    # target_brightness: mức độ sáng chuẩn (0-255) muốn normalize về.
    #   128 = "trung tính hoàn toàn", 115-130 thường là khoảng tốt nhất.
    # gamma_min / gamma_max: clamp để tránh correction cực đoan.
    #   min=0.4 → không brightening quá 2.5x; max=3.0 → không darkening quá mạnh.
    'agc_target'           : 128,     # mức sáng chuẩn (target brightness)
    'agc_gamma_min'        : 0.4,     # clamp dưới (ảnh quá tối)
    'agc_gamma_max'        : 3.0,     # clamp trên (ảnh quá chói)
    # Jitter range cho training (để model không overfit vào đúng target=128)
    'agc_jitter'           : 25,      # target ± 25 → [103, 153]

    # Phase 1: Head Warm-up
    'p1_epochs'       : 5,
    'p1_lr'           : 1e-3,
    'p1_img_size'     : 360,
    'p1_batch'        : _B['p1'] * _SCALE,
    'p1_label_smooth' : 0.0,
    'p1_mixup'        : 0.0,
    'p1_cutmix'       : 0.0,
    'p1_aug'          : 'light',

    # Phase 2: Progressive Unfreeze
    'p2_epochs'       : 12,
    'p2_lr_head'      : 4e-4,
    'p2_lr_backbone'  : 4e-5,
    'p2_img_size'     : _P2_IMG,
    'p2_batch'        : _B['p2'] * _SCALE,
    'p2_label_smooth' : 0.05,
    'p2_mixup'        : 0.1,
    'p2_cutmix'       : 0.0,
    'p2_aug'          : 'medium',

    # Phase 3: Full Fine-tune
    'p3_epochs'       : 45,
    'p3_lr_head'      : 2.5e-4,
    'p3_lr_backbone'  : 8e-6,
    'p3_img_size'     : 384,
    'p3_batch'        : _B['p3'] * _SCALE,
    'p3_label_smooth' : 0.08,
    'p3_mixup'        : 0.2,
    'p3_cutmix'       : 0.2,
    'p3_aug'          : 'heavy',
    'p3_patience'     : 18,

    # Phase 4: SWA
    'p4_epochs'       : 12,
    'p4_lr'           : 3e-5,
    'p4_img_size'     : 384,
    'p4_batch'        : _B['p4'] * _SCALE,
    'p4_label_smooth' : 0.03,
    'p4_mixup'        : 0.0,
    'p4_cutmix'       : 0.0,
    'p4_aug'          : 'medium',

    # Phase 5: Lighting Domain Adaptation
    'p5_epochs'       : 15,
    'p5_lr_head'      : 5e-5,
    'p5_lr_backbone'  : 2e-6,
    'p5_img_size'     : 384,
    'p5_label_smooth' : 0.03,
    'p5_mixup'        : 0.1,
    'p5_cutmix'       : 0.0,
    'p5_aug'          : 'lighting',

    # ── Phase 6: Real-World Fine-Tuning ─────────────────────────────────
    # LR SIÊU NHỎ — nguyên tắc sống còn để tránh Catastrophic Forgetting.
    'p6_base_ckpt'    : 'best_model.pth',  # file .pth tốt nhất từ Phase 5
    'p6_epochs'       : 15,
    'p6_lr_head'      : 3e-5,              # 1/8 so với Phase 3
    'p6_lr_backbone'  : 1e-6,              # 1/8 so với Phase 3
    'p6_img_size'     : 384,
    'p6_batch'        : _B['p3'] * _SCALE,
    'p6_label_smooth' : 0.02,
    'p6_mixup'        : 0.1,
    'p6_cutmix'       : 0.0,
    'p6_aug'          : 'lighting',        # aug mạnh, tập trung IoT lighting
    'p6_patience'     : 8,                 # early stop patience
    'p6_ema_decay'    : 0.9998,            # EMA cao hơn = ổn định hơn
}

def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG['seed'])
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f'Classes ({NUM_CLASSES}): {CLASSES}')
print(f'Output: {CONFIG["output_dir"]}')
print(f'AGC: target={CONFIG["agc_target"]}  '
      f'clip=[{CONFIG["agc_gamma_min"]}, {CONFIG["agc_gamma_max"]}]  '
      f'jitter=±{CONFIG["agc_jitter"]}')
print(f'Phase 6: LR head={CONFIG["p6_lr_head"]:.0e}  '
      f'backbone={CONFIG["p6_lr_backbone"]:.0e}  '
      f'epochs={CONFIG["p6_epochs"]}  '
      f'base_ckpt={CONFIG["p6_base_ckpt"]}')


In [ ]:
# ============================================================
# CELL 3 - DATASET & AUGMENTATION  (v5 - Adaptive Gamma)
# ============================================================
import math as _math

# ══════════════════════════════════════════════════════════════════════════════
#  ADAPTIVE GAMMA CORRECTION  —  Giải thích đầy đủ
# ══════════════════════════════════════════════════════════════════════════════
#
#  Vấn đề với gamma cố định (v4):
#    - Gamma = 1.2 kéo sáng MỌI ảnh như nhau, kể cả ảnh đã sáng đủ rồi.
#    - Ánh sáng mặt trời buổi trưa (mean≈200) → ảnh càng bị chói hơn.
#    - Đêm tối (mean≈30) → 1.2 chưa đủ để kéo sáng.
#
#  Giải pháp Adaptive Gamma:
#    Bước 1: Đo độ sáng trung bình của ảnh (kênh V trong không gian màu HSV).
#            Kênh V = max(R,G,B) / 255 — đây là luminance perceptual chính xác nhất,
#            không bị ảnh hưởng bởi màu sắc (hue/saturation).
#
#    Bước 2: Tính gamma cần thiết để đưa mean_V về target (mặc định 128):
#              γ = log(target/255) / log(mean_V/255)
#            Nếu mean_V < target → log(mean_V/255) âm hơn log(target/255)
#              → γ < 1 → ảnh được kéo SÁNG (đúng, vì ảnh đang tối).
#            Nếu mean_V > target → γ > 1 → ảnh được DÌM (đúng, vì ảnh đang chói).
#
#    Bước 3: Build Lookup Table (256 giá trị) và apply — O(1), vài micro-giây.
#
#    Bước 4 (chỉ training): Jitter target ± N để model học robustness với
#            các mức sáng hơi khác nhau thay vì overfit vào đúng target=128.
#
#  Sơ đồ các trường hợp:
#    mean=30  (đêm)    → γ≈0.45  → kéo sáng mạnh
#    mean=80  (trong nhà tối) → γ≈0.78  → kéo sáng nhẹ
#    mean=128 (lý tưởng)  → γ=1.00  → pass-through
#    mean=180 (nắng)   → γ≈1.65  → dìm sáng vừa
#    mean=220 (nắng gắt) → γ≈2.40  → dìm sáng mạnh
# ══════════════════════════════════════════════════════════════════════════════

class AdaptiveGammaCorrection:
    """
    Adaptive Gamma Correction động theo từng ảnh.

    Dùng cho VALIDATION và INFERENCE (target cố định → kết quả deterministic).

    Args:
        target      : độ sáng chuẩn muốn normalize về (0-255), mặc định 128.
        gamma_min   : clamp dưới — giới hạn kéo sáng tối đa (mặc định 0.4).
        gamma_max   : clamp trên — giới hạn dìm sáng tối đa (mặc định 3.0).
    """
    def __init__(self, target: int = 128,
                 gamma_min: float = 0.4,
                 gamma_max: float = 3.0):
        self.target    = int(target)
        self.gamma_min = gamma_min
        self.gamma_max = gamma_max
        # Index array dùng để build LUT nhanh
        self._idx = np.arange(256, dtype=np.float64) / 255.0

    def _compute_gamma(self, mean_v: float) -> float:
        """
        Công thức: γ = log(target/255) / log(mean_V/255)

        Edge cases:
          - mean_v quá nhỏ (<8): clamp để tránh gamma = ∞
          - mean_v quá lớn (>247): clamp để tránh division by zero
          - mean_v xấp xỉ target: trả về 1.0 (không cần sửa)
        """
        mean_v  = float(np.clip(mean_v, 8.0, 247.0))
        target  = float(np.clip(self.target, 8.0, 247.0))

        log_mean   = _math.log(mean_v   / 255.0)
        log_target = _math.log(target   / 255.0)

        # Nếu mean gần target (sai lệch < 3%): không cần sửa
        if abs(log_mean - log_target) < 0.03:
            return 1.0

        gamma = log_target / log_mean
        return float(np.clip(gamma, self.gamma_min, self.gamma_max))

    def _build_lut(self, gamma: float) -> np.ndarray:
        """Build Lookup Table 256 giá trị dựa trên gamma."""
        lut = np.power(self._idx, gamma) * 255.0
        return lut.clip(0, 255).astype(np.uint8)

    def _get_mean_v(self, arr: np.ndarray) -> float:
        """
        Tính độ sáng trung bình trên kênh V (Value) của HSV.
        V = max(R, G, B) — đây là luminance perceptual chính xác.
        Nhanh hơn convert PIL sang HSV và không cần import thêm thư viện.
        """
        return arr.max(axis=2).mean().item()

    def compute_gamma_for_image(self, img: Image.Image) -> float:
        """Public method để debug / log gamma của từng ảnh."""
        arr = np.asarray(img, dtype=np.uint8)
        return self._compute_gamma(self._get_mean_v(arr))

    def __call__(self, img: Image.Image) -> Image.Image:
        arr    = np.asarray(img, dtype=np.uint8)
        mean_v = self._get_mean_v(arr)
        gamma  = self._compute_gamma(mean_v)

        if abs(gamma - 1.0) < 0.02:
            return img                         # pass-through, không cần xử lý

        lut = self._build_lut(gamma)
        out = lut[arr]                         # vectorised index — cực nhanh O(1)
        return Image.fromarray(out.astype(np.uint8))

    def __repr__(self):
        return (f'AdaptiveGammaCorrection(target={self.target}, '
                f'clip=[{self.gamma_min}, {self.gamma_max}])')


class RandomAdaptiveGammaJitter:
    """
    Phiên bản Training của Adaptive Gamma Correction.

    Thêm jitter ngẫu nhiên vào target brightness để model học robustness:
      target_actual = target_base ± uniform(0, jitter)
      → [target - jitter, target + jitter]

    Tại sao cần jitter?
      Nếu val/inference luôn normalize về ĐÚNG 128, mà training cũng về 128,
      model sẽ overfit: "ảnh nào cũng có mean=128 → chỉ cần học màu sắc/hình dạng".
      Với jitter=25, model thấy ảnh có mean từ 103-153 → học robust hơn.
    """
    def __init__(self, target: int = 128, jitter: int = 25,
                 gamma_min: float = 0.4, gamma_max: float = 3.0,
                 p: float = 0.85):
        self.target    = target
        self.jitter    = jitter
        self.gamma_min = gamma_min
        self.gamma_max = gamma_max
        self.p         = p     # xác suất áp dụng (để 15% ảnh không bị sửa)
        self._base_agc = AdaptiveGammaCorrection(target, gamma_min, gamma_max)
        self._idx      = np.arange(256, dtype=np.float64) / 255.0

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.rand() > self.p:
            return img

        # Jitter target
        jittered_target = int(np.clip(
            self.target + np.random.randint(-self.jitter, self.jitter + 1),
            max(self.target - self.jitter, 10),
            min(self.target + self.jitter, 245),
        ))
        agc = AdaptiveGammaCorrection(jittered_target, self.gamma_min, self.gamma_max)
        return agc(img)

    def __repr__(self):
        return (f'RandomAdaptiveGammaJitter(target={self.target}±{self.jitter}, '
                f'p={self.p})')


# ── Lighting Augmentation Transforms ────────────────────────────────────────

class RandomShadowInjection:
    """
    Mô phỏng bóng đổ cứng (hard shadow) — bóng cành cây / tay người / thanh sắt.
    Tạo vệt đen hình chữ nhật ngẫu nhiên theo chiều ngang hoặc dọc.

    Quan trọng: Áp dụng SAU AdaptiveGamma (shadow là nhiễu residual sau normalize).
    """
    def __init__(self, p: float = 0.4, dark_lo: float = 0.3, dark_hi: float = 0.65):
        self.p       = p
        self.dark_lo = dark_lo
        self.dark_hi = dark_hi

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.rand() > self.p:
            return img

        arr = np.array(img, dtype=np.float32)
        h, w = arr.shape[:2]

        if np.random.rand() < 0.5:
            y1 = np.random.randint(0, max(1, h // 2))
            y2 = np.random.randint(h // 2, h)
            x1, x2 = 0, w
        else:
            x1 = np.random.randint(0, max(1, w // 2))
            x2 = np.random.randint(w // 2, w)
            y1, y2 = 0, h

        factor = np.random.uniform(self.dark_lo, self.dark_hi)
        arr[y1:y2, x1:x2] *= factor
        return Image.fromarray(arr.clip(0, 255).astype(np.uint8))

    def __repr__(self):
        return f'RandomShadowInjection(p={self.p})'


class RandomHardLight:
    """
    Mô phỏng ánh sáng chói cục bộ (hard light flare) — nắng gắt chiếu thẳng
    hoặc đèn đường gần camera Raspberry Pi.

    Tạo vùng sáng ellipse với Gaussian mask viền mềm.
    Áp dụng SAU AdaptiveGamma — đây là nhiễu residual không normalize được.
    """
    def __init__(self, p: float = 0.3, intensity_lo: float = 1.4, intensity_hi: float = 2.5):
        self.p            = p
        self.intensity_lo = intensity_lo
        self.intensity_hi = intensity_hi

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.rand() > self.p:
            return img

        arr = np.array(img, dtype=np.float32)
        h, w = arr.shape[:2]
        cx = np.random.randint(w // 4, 3 * w // 4)
        cy = np.random.randint(h // 4, 3 * h // 4)
        rx = np.random.randint(w // 6, w // 3)
        ry = np.random.randint(h // 6, h // 3)

        yy, xx = np.ogrid[:h, :w]
        mask = np.exp(-(((xx - cx) / max(rx, 1)) ** 2 + ((yy - cy) / max(ry, 1)) ** 2))
        intensity = np.random.uniform(self.intensity_lo, self.intensity_hi)
        arr += mask[:, :, np.newaxis] * intensity * 60
        return Image.fromarray(arr.clip(0, 255).astype(np.uint8))

    def __repr__(self):
        return f'RandomHardLight(p={self.p})'


# ── Dataset ──────────────────────────────────────────────────────────────────

class WasteDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with Image.open(path) as img:
            img = img.convert('RGB')
            if self.transform:
                img = self.transform(img)
        return img, label


def _resolve_data_root(data_dir):
    data_path = Path(data_dir)
    fallback_candidates = [
        Path('./Model/DataSet/Data'), Path('./Model/DataSet'),
        Path('./DataSet/Data'), Path('./DataSet'),
        Path('./dataset'), Path('./data'),
    ]
    if not data_path.exists():
        for p in fallback_candidates:
            if p.exists():
                data_path = p; break
    if not data_path.exists():
        raise FileNotFoundError(f'Data dir not found: {data_dir}')
    nested = data_path / 'Data'
    if nested.exists() and nested.is_dir():
        data_path = nested
    return data_path


def _canonical_from_folder(folder_name):
    return CANONICAL_BY_NORM.get(_norm_name(folder_name))


def load_samples(data_dir):
    samples   = []
    data_path = _resolve_data_root(data_dir)
    print(f'[INFO] Data root: {data_path}')

    folder_map       = {f.name: f for f in data_path.iterdir() if f.is_dir()}
    canonical_to_dir = {}
    for fn, fp in folder_map.items():
        cn = _canonical_from_folder(fn)
        if cn and cn not in canonical_to_dir:
            canonical_to_dir[cn] = fp
    for cls_name in CLASSES:
        exact = data_path / cls_name
        if exact.exists() and exact.is_dir():
            canonical_to_dir[cls_name] = exact

    missing = [c for c in CLASSES if c not in canonical_to_dir]
    if missing:
        raise RuntimeError(f'Missing class folders: {missing}')

    for cls_name in CLASSES:
        cls_dir = canonical_to_dir[cls_name]; cls_idx = CLASS2IDX[cls_name]; cnt = 0
        for ext in ('*.jpg','*.jpeg','*.png','*.bmp','*.webp',
                    '*.JPG','*.JPEG','*.PNG','*.BMP','*.WEBP'):
            for p in cls_dir.glob(ext):
                samples.append((str(p), cls_idx)); cnt += 1
        print(f'  {cls_name:<20}: {cnt} images')

    print(f'  Total: {len(samples)}')
    if samples:
        labels = np.array([s[1] for s in samples])
        binc   = np.bincount(labels, minlength=NUM_CLASSES)
        print(f'  Imbalance ratio: {(binc.max()/max(binc.min(),1)):.2f}x')
    return samples


def train_val_split_stratified(samples, val_ratio=0.2, seed=42):
    labels = np.array([s[1] for s in samples])
    uniq, cnts = np.unique(labels, return_counts=True)
    if len(uniq) != NUM_CLASSES:
        raise RuntimeError(f'Found {len(uniq)} classes, expected {NUM_CLASSES}.')
    if np.any(cnts < 2):
        print('[WARN] Some classes <2 — random split.')
        rng = np.random.default_rng(seed); idx = np.arange(len(samples)); rng.shuffle(idx)
        val_n = min(max(NUM_CLASSES, int(len(samples)*val_ratio)), len(samples)-1)
        return [samples[i] for i in idx[val_n:]], [samples[i] for i in idx[:val_n]]
    eff = max(val_ratio, NUM_CLASSES/len(samples))
    sss = StratifiedShuffleSplit(1, test_size=eff, random_state=seed)
    ti, vi = next(sss.split(np.zeros(len(samples)), labels))
    return [samples[i] for i in ti], [samples[i] for i in vi]


# ── Transform Factory ─────────────────────────────────────────────────────────

def get_transforms(img_size: int, aug_level: str = 'light', config: dict = None):
    """
    Tạo transform pipeline — v7: RandomResizedCrop cho training.

    Chiến lược:
      Training (light/medium/heavy/lighting):
        AGC jitter → RandomResizedCrop(scale, ratio) → augment → Normalize
        - AGC TRƯỚC crop để đo mean brightness trên toàn ảnh gốc (chính xác hơn).
        - RandomResizedCrop một mình thay thế Resize+RandomCrop:
            crop ngẫu nhiên 40-100% diện tích ảnh, từ BẤT KỲ vị trí nào,
            resize về img_size với BICUBIC — không méo aspect ratio của object.
        - Object ở góc? ✓  Object ở viền? ✓  Object ở giữa? ✓
        - Tiêu chuẩn của Inception, EfficientNet, MobileNet, ResNet.

      Validation / Inference:
        Resize(int, ngắn nhất) → AGC deterministic → CenterCrop → Normalize
        - Tại sao CenterCrop? Val pipeline mirror INFERENCE thực tế:
            Camera Raspberry Pi CỐ ĐỊNH → thùng rác luôn ở trung tâm frame.
            Val đo accuracy ở điều kiện inference, không phải dataset distribution.
        - CenterCrop đúng cho use case này.

    aug_level:
      'val'      — Resize(int) + AGC + CenterCrop  [val & inference]
      'light'    — AGC + RandomResizedCrop(60-100%) + flip nhẹ
      'medium'   — AGC + RandomResizedCrop(50-100%) + augment vừa
      'heavy'    — AGC + RandomResizedCrop(40-100%) + full lighting aug
      'lighting' — AGC + RandomResizedCrop(50-100%) + lighting cường độ cao
    """
    cfg = config or CONFIG
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    agc_target   = cfg.get('agc_target',    128)
    agc_min      = cfg.get('agc_gamma_min', 0.4)
    agc_max      = cfg.get('agc_gamma_max', 3.0)
    agc_jitter   = cfg.get('agc_jitter',    25)

    # AGC deterministic — dùng cho val/inference
    agc_det = AdaptiveGammaCorrection(agc_target, agc_min, agc_max)
    # AGC với jitter — dùng cho training
    agc_jit = RandomAdaptiveGammaJitter(agc_target, agc_jitter, agc_min, agc_max, p=0.85)

    # Kích thước resize cạnh ngắn cho val (margin 15% — chuẩn ImageNet)
    val_resize = int(img_size * 1.15)

    # ── Validation / Inference ────────────────────────────────────────────
    # Tại sao vẫn dùng CenterCrop cho val/inference?
    #   Val pipeline phải mirror INFERENCE thực tế — không phải dataset distribution.
    #   Camera Raspberry Pi được gắn cố định hướng vào thùng rác
    #   → object luôn nằm ở trung tâm frame khi inference thực tế.
    #   → CenterCrop đúng cho inference.
    #   → Val pipeline dùng CenterCrop để đo accuracy đúng với điều kiện thực.
    #
    # Resize(int) giữ aspect ratio → ảnh không bị méo trước khi crop.
    # val_resize = img_size * 1.15 để có thêm padding xung quanh trước CenterCrop.
    if aug_level == 'val':
        return transforms.Compose([
            transforms.Resize(val_resize),       # << int → giữ aspect ratio
            agc_det,                             # << AGC trước crop (đo mean toàn ảnh)
            transforms.CenterCrop(img_size),     # << camera Pi cố định → đúng
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

    # ── Light (Phase 1 - Head warm-up) ───────────────────────────────────
    # RandomResizedCrop thay thế Resize + RandomCrop:
    #   - Crop ngẫu nhiên vùng 60-100% diện tích ảnh, từ BẤT KỲ vị trí nào
    #   - Resize vùng crop về img_size (không méo aspect ratio của object)
    #   - Object ở góc? ✓  Object ở viền? ✓  Object ở giữa? ✓
    if aug_level == 'light':
        return transforms.Compose([
            agc_jit,                             # << AGC trước crop (mean toàn ảnh)
            transforms.RandomResizedCrop(        # << một transform thay thế Resize+Crop
                img_size,
                scale=(0.6, 1.0),               # crop 60-100% diện tích
                ratio=(3/4, 4/3),               # aspect ratio hợp lý
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(0.5),
            transforms.ColorJitter(0.15, 0.15, 0.15, 0.03),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

    # ── Medium (Phase 2 & SWA) ────────────────────────────────────────────
    if aug_level == 'medium':
        return transforms.Compose([
            agc_jit,
            transforms.RandomResizedCrop(
                img_size,
                scale=(0.5, 1.0),               # crop rộng hơn → scale diversity
                ratio=(3/4, 4/3),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.05),
            transforms.ColorJitter(0.25, 0.25, 0.25, 0.04),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.1, scale=(0.02, 0.08)),
        ])

    # ── Heavy (Phase 3 - Full fine-tune) ─────────────────────────────────
    # Thứ tự cố ý:
    #   1. AGC jitter TRƯỚC RandomResizedCrop → đo mean trên toàn ảnh gốc
    #   2. RandomResizedCrop → vị trí + scale ngẫu nhiên, không méo object
    #   3. Shadow / HardLight → inject residual noise CỤC BỘ sau khi đã crop
    #   4. ColorJitter mạnh → màu sắc / contrast
    #   5. Geometric aug → xoay, perspective
    if aug_level == 'heavy':
        return transforms.Compose([
            agc_jit,                             # Step 1: normalize toàn ảnh
            transforms.RandomResizedCrop(        # Step 2: vị trí & scale ngẫu nhiên
                img_size,
                scale=(0.4, 1.0),               # crop 40-100% → scale diversity mạnh nhất
                ratio=(3/4, 4/3),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.1),
            RandomShadowInjection(p=0.45, dark_lo=0.3,  dark_hi=0.65),   # Step 3a
            RandomHardLight(p=0.30,      intensity_lo=1.4, intensity_hi=2.5),  # Step 3b
            transforms.ColorJitter(brightness=0.5, contrast=0.5,         # Step 4
                                   saturation=0.4, hue=0.08),
            transforms.RandomGrayscale(p=0.05),
            transforms.RandomRotation(20),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
            transforms.RandAugment(num_ops=2, magnitude=7),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
        ])

    # ── Lighting (Phase 5 - Domain Adaptation) ────────────────────────────
    # Giảm scale range (0.5-1.0 thay vì 0.4) để crop không quá nhỏ,
    # đảm bảo object vẫn đủ rõ sau khi inject shadow/hardlight.
    if aug_level == 'lighting':
        return transforms.Compose([
            agc_jit,
            transforms.RandomResizedCrop(
                img_size,
                scale=(0.5, 1.0),
                ratio=(3/4, 4/3),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(0.5),
            RandomShadowInjection(p=0.55, dark_lo=0.25, dark_hi=0.70),
            RandomHardLight(p=0.45,      intensity_lo=1.5, intensity_hi=3.0),
            transforms.ColorJitter(brightness=0.6, contrast=0.55,
                                   saturation=0.5, hue=0.10),
            transforms.RandomGrayscale(p=0.08),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.15, scale=(0.02, 0.10)),
        ])

    return get_transforms(img_size, 'heavy', config)   # fallback


def make_weighted_sampler(samples, power=0.7):
    counts = np.zeros(NUM_CLASSES, dtype=np.float32)
    for _, lbl in samples:
        counts[lbl] += 1
    wpc = 1.0 / np.power(np.maximum(counts, 1), power)
    sw  = torch.tensor([wpc[lbl] for _, lbl in samples], dtype=torch.float32)
    return WeightedRandomSampler(sw, len(sw), replacement=True)


def make_dataloaders(trn_samples, val_samples, img_size, batch_size,
                     aug_level, num_workers, device_type,
                     old_trn_dl=None, old_val_dl=None,
                     config=None):
    if old_trn_dl is not None: del old_trn_dl
    if old_val_dl is not None: del old_val_dl
    gc.collect()

    cfg    = config or CONFIG
    trn_tf = get_transforms(img_size, aug_level, cfg)
    val_tf = get_transforms(img_size, 'val',     cfg)   # << AGC deterministic

    trn_ds  = WasteDataset(trn_samples, trn_tf)
    val_ds  = WasteDataset(val_samples, val_tf)
    sampler = make_weighted_sampler(trn_samples, cfg.get('sampler_power', 0.7))
    pin     = (device_type == 'cuda' and num_workers > 0)

    trn_kw = dict(batch_size=batch_size, sampler=sampler,
                  num_workers=num_workers, pin_memory=pin, drop_last=True)
    val_bs = batch_size if img_size >= 384 else batch_size * 2
    val_kw = dict(batch_size=val_bs, shuffle=False,
                  num_workers=num_workers, pin_memory=pin)

    if num_workers > 0:
        trn_kw.update(persistent_workers=False, prefetch_factor=2)
        val_kw.update(persistent_workers=False, prefetch_factor=2)

    trn_dl = DataLoader(trn_ds, **trn_kw)
    val_dl = DataLoader(val_ds, **val_kw)
    print('Dataloaders ready.')
    return trn_dl, val_dl


# ── Quick sanity check ────────────────────────────────────────────────────────
def _demo_agc():
    agc = AdaptiveGammaCorrection(target=128)
    scenarios = [
        ('Đêm tối (mean=30)',      30),
        ('Trong nhà (mean=80)',    80),
        ('Lý tưởng (mean=128)',   128),
        ('Nắng buổi chiều (mean=180)', 180),
        ('Nắng trưa (mean=220)',  220),
    ]
    print('\n  AGC Gamma theo độ sáng thực tế:')
    print(f'  {"Scenario":<35} {"Mean V":>8} {"Gamma":>8} {"Action":>20}')
    print('  ' + '-'*75)
    for label, mean_v in scenarios:
        g = agc._compute_gamma(float(mean_v))
        action = 'KÉOSÁNG' if g < 0.95 else ('DÌM' if g > 1.05 else 'PASS-THROUGH')
        print(f'  {label:<35} {mean_v:>8} {g:>8.3f} {action:>20}')
    print()

_demo_agc()
print('Dataset & Augmentation module ready (v5 — Adaptive Gamma).')
print('  Transforms: AdaptiveGammaCorrection | RandomAdaptiveGammaJitter | '
      'RandomShadowInjection | RandomHardLight')


In [ ]:
# ============================================================
# CELL 4 - MODEL, LOSS, UTILITIES
# ============================================================

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(
            x.clamp(min=self.eps).pow(self.p), (1,1)
        ).pow(1.0 / self.p)


class WasteDetector(nn.Module):
    def __init__(self, num_classes, pretrained=True, dropout=0.4):
        super().__init__()
        weights       = MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None
        base          = mobilenet_v3_large(weights=weights)
        self.features = base.features
        self.gem_pool = GeM(p=3)
        in_f          = base.classifier[0].in_features  # 960

        self.classifier = nn.Sequential(
            nn.Linear(in_f, 512),
            nn.BatchNorm1d(512),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(256, num_classes),
        )
        self.objectness = nn.Sequential(
            nn.Linear(in_f, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        feat   = self.features(x)
        feat   = self.gem_pool(feat).flatten(1)
        logits = self.classifier(feat)
        obj    = self.objectness(feat)
        return logits, obj


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        if alpha is not None:
            self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None

    def forward(self, logits, targets):
        C = logits.size(1)
        if self.label_smoothing > 0:
            smooth   = self.label_smoothing / C
            one_hot  = torch.zeros_like(logits).scatter(1, targets.unsqueeze(1).long(), 1)
            one_hot  = one_hot * (1 - self.label_smoothing) + smooth
            log_prob = F.log_softmax(logits, dim=1)
            ce       = -(one_hot * log_prob).sum(1)
        else:
            ce = F.cross_entropy(logits, targets, reduction='none')
        pt      = torch.exp(-ce.clamp(min=0.0, max=100.0))
        focal_w = (1 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets.view(-1).long())
            focal_w = focal_w * alpha_t
        return (focal_w * ce).mean()


def compute_class_alpha(samples, power=0.7):
    counts = np.zeros(NUM_CLASSES, dtype=np.float32)
    for _, lbl in samples:
        counts[lbl] += 1
    alpha = 1.0 / np.power(np.maximum(counts, 1), power)
    return (alpha / alpha.sum() * NUM_CLASSES).tolist()


def mixup_data(x, y, alpha=0.2):
    if alpha <= 0: return x, y, y, 1.0
    lam  = max(np.random.beta(alpha, alpha), 1.0 - np.random.beta(alpha, alpha))
    ridx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[ridx], y, y[ridx], lam


def cutmix_data(x, y, alpha=1.0):
    if alpha <= 0: return x, y, y, 1.0
    lam  = max(np.random.beta(alpha, alpha), 1.0 - np.random.beta(alpha, alpha))
    ridx = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape
    cr   = math.sqrt(1.0 - lam)
    cw, ch = int(W*cr), int(H*cr)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1,x2 = max(cx-cw//2,0), min(cx+cw//2,W)
    y1,y2 = max(cy-ch//2,0), min(cy+ch//2,H)
    mx = x.clone(); mx[:,:,y1:y2,x1:x2] = x[ridx,:,y1:y2,x1:x2]
    lam = 1 - (x2-x1)*(y2-y1)/(W*H)
    return mx, y, y[ridx], lam


def mixed_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1-lam) * criterion(pred, yb)


class ModelEMA:
    def __init__(self, model, decay=0.9995):
        self.ema   = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model, epoch=0):
        d = min(self.decay, (1 + epoch) / (10 + epoch))
        for ep, mp in zip(self.ema.parameters(), model.parameters()):
            ep.data.mul_(d).add_(mp.data, alpha=1-d)
        for eb, mb in zip(self.ema.buffers(), model.buffers()):
            eb.data.copy_(mb.data)


def get_backbone_groups(model):
    f = model.features
    return [list(f[0:4].parameters()), list(f[4:7].parameters()),
            list(f[7:13].parameters()), list(f[13:].parameters())]

def freeze_backbone(model):
    for p in model.features.parameters():
        p.requires_grad = False

def unfreeze_groups(model, idxs):
    for g in idxs:
        for p in get_backbone_groups(model)[g]:
            p.requires_grad = True

def make_param_groups(model, lr_head, lr_bb):
    pgs = []
    for gi, params in enumerate(get_backbone_groups(model)):
        trainable = [p for p in params if p.requires_grad]
        if trainable:
            scale = max(2**(gi-3), 0.125)
            pgs.append({'params': trainable, 'lr': lr_bb * scale, 'name': f'bb_g{gi}'})
    head_p = (list(model.classifier.parameters()) +
              list(model.objectness.parameters()) +
              list(model.gem_pool.parameters()))
    pgs.append({'params': [p for p in head_p if p.requires_grad],
                'lr': lr_head, 'name': 'head'})
    return pgs

print('Model & Loss module ready.')


In [ ]:
# ============================================================
# CELL 5 – TRAIN / VALIDATE LOOPS
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, device,
                    mixup=0.0, cutmix=0.0, scheduler=None,
                    scaler=None, epoch=0):
    model.train()
    total_loss = correct = total = 0
    use_amp = (device.type == 'cuda' and scaler is not None and scaler.is_enabled())

    pbar = tqdm(loader, desc='  Train', leave=False, dynamic_ncols=True)
    for imgs, labels in pbar:
        min_label = labels.min().item(); max_label = labels.max().item()
        if min_label < 0 or max_label >= NUM_CLASSES:
            raise ValueError(f'Invalid labels: min={min_label}, max={max_label}')

        _nb = (device.type == 'cuda')
        imgs   = imgs.to(device, non_blocking=_nb)
        labels = labels.to(device, non_blocking=_nb)

        use_cut = cutmix > 0 and np.random.rand() < 0.5
        use_mix = mixup  > 0 and not use_cut
        if use_cut:   imgs, la, lb, lam = cutmix_data(imgs, labels, cutmix)
        elif use_mix: imgs, la, lb, lam = mixup_data(imgs, labels, mixup)
        else:         la, lb, lam = labels, labels, 1.0

        _amp_ctx = (torch.autocast(device_type='cuda', enabled=True)
                    if use_amp else contextlib.nullcontext())
        with _amp_ctx:
            logits, _ = model(imgs)
            loss = (mixed_criterion(criterion, logits, la, lb, lam)
                    if lam < 1.0 else criterion(logits, labels))

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.detach().argmax(1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        pbar.set_postfix(loss=f'{loss.item():.3f}')
        del imgs, labels, logits, loss

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device, return_preds=False):
    model.eval()
    total_loss = correct = total = 0
    cls_correct = defaultdict(int); cls_total = defaultdict(int)
    use_amp     = (device.type == 'cuda')
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        min_label = labels.min().item(); max_label = labels.max().item()
        if min_label < 0 or max_label >= NUM_CLASSES:
            raise ValueError(f'Invalid labels in val: min={min_label}, max={max_label}')

        _nb = (device.type == 'cuda')
        imgs   = imgs.to(device, non_blocking=_nb)
        labels = labels.to(device, non_blocking=_nb)

        _amp_ctx = (torch.autocast(device_type='cuda', enabled=True)
                    if use_amp else contextlib.nullcontext())
        with _amp_ctx:
            logits, _ = model(imgs)
            loss      = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        preds_cpu  = logits.detach().argmax(1).cpu()
        labels_cpu = labels.cpu()
        correct   += (preds_cpu == labels_cpu).sum().item()
        total     += labels_cpu.size(0)

        for p, l in zip(preds_cpu.numpy(), labels_cpu.numpy()):
            cls_total[l] += 1; cls_correct[l] += int(p == l)

        if return_preds:
            all_preds.extend(preds_cpu.numpy())
            all_labels.extend(labels_cpu.numpy())
        del imgs, labels, logits, loss

    per_cls = {IDX2CLASS[k]: cls_correct[k]/max(cls_total[k],1)
               for k in sorted(cls_total)}
    if return_preds:
        return total_loss/total, correct/total, per_cls, all_preds, all_labels
    return total_loss/total, correct/total, per_cls


def _track(h, tl, ta, vl, va, ea, phase, lr):
    h['train_loss'].append(tl); h['train_acc'].append(ta)
    h['val_loss'].append(vl);   h['val_acc'].append(va)
    h['ema_val_acc'].append(ea); h['phase'].append(phase)
    h['lr'].append(lr)


def _maybe_save(model, ema, val_acc, ema_acc, best_acc,
                best_w, best_ew, path, config, epoch):
    eff = max(val_acc, ema_acc)
    if eff > best_acc:
        best_acc = eff
        best_w   = copy.deepcopy(model.state_dict())
        best_ew  = copy.deepcopy(ema.ema.state_dict())
        torch.save({'epoch': epoch, 'model_state': best_w, 'ema_state': best_ew,
                    'val_acc': best_acc, 'classes': CLASSES,
                    'img_size': config['img_size'],
                    'gamma': config.get('gamma_correction', 1.2)},
                   path)
        print(f'  >> Saved  acc={best_acc:.4f}')
    return best_acc, best_w, best_ew


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()
    elif hasattr(torch.mps, 'empty_cache'):
        torch.mps.empty_cache()


def gpu_mem_str():
    if not torch.cuda.is_available(): return ''
    used  = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return f' [VRAM {used:.1f}/{total:.1f}GB]'


print('Training loops ready.')


In [ ]:
# ============================================================
# CELL 6 - VISUALIZATION
# ============================================================

def plot_history(history, out_dir):
    fig, axes = plt.subplots(1, 3, figsize=(21, 5))
    ep = range(1, len(history['train_loss']) + 1)
    axes[0].plot(ep, history['train_loss'], label='Train')
    axes[0].plot(ep, history['val_loss'],   label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(ep, history['train_acc'], label='Train')
    axes[1].plot(ep, history['val_acc'],   label='Val')
    if any(v > 0 for v in history['ema_val_acc']):
        axes[1].plot(ep, history['ema_val_acc'], label='EMA', ls='--', alpha=0.7)
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)

    axes[2].plot(ep, history['lr'], color='tab:orange')
    axes[2].set_title('LR'); axes[2].set_yscale('log'); axes[2].grid(True)

    for i in range(1, len(history['phase'])):
        if history['phase'][i] != history['phase'][i-1]:
            for ax in axes:
                ax.axvline(i+1, color='gray', ls=':', alpha=0.5)

    plt.tight_layout()
    path = os.path.join(out_dir, 'training_history.png')
    plt.savefig(path, dpi=120); plt.close(); plt.clf()
    print(f'  >> Plot: {path}')


def plot_confusion_matrix(preds, labels, out_dir):
    label_ids = list(range(NUM_CLASSES))
    cm = confusion_matrix(labels, preds, labels=label_ids)
    row_sum = cm.sum(1, keepdims=True)
    cm_norm = np.zeros_like(cm, dtype=np.float32)
    np.divide(cm.astype(np.float32), row_sum, out=cm_norm, where=row_sum != 0)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
    axes[0].set_title('Confusion (counts)'); axes[0].tick_params(axis='x', rotation=45)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
    axes[1].set_title('Confusion (normalized)'); axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    path = os.path.join(out_dir, 'confusion_matrix.png')
    plt.savefig(path, dpi=120); plt.close(); plt.clf()
    print(f'  >> Confusion: {path}')
    print(classification_report(labels, preds, labels=label_ids,
                                 target_names=CLASSES, zero_division=0))

print('Visualization ready.')


In [ ]:
# ============================================================
# CELL 7 - DATA PREPARATION  (Phase 6)
# ============================================================
# Hỗ trợ 2 chế độ:
#
# Mode A — Dataset_v2 dạng FLAT (tất cả trong 1 thư mục, auto-split):
#   Dataset_v2/
#     Battery/       ← gồm ảnh cũ + ảnh real mới
#     Biological/
#     ...
#   → set DATA_MODE = 'flat'
#   → set DATA_DIR_V2 = 'path/to/Dataset_v2'
#
# Mode B — Dataset_v2 đã chia sẵn train/val:
#   Dataset_v2/
#     train/
#       Battery/   Biological/   ...
#     val/
#       Battery/   Biological/   ...
#   → set DATA_MODE = 'split'
#   → set DATA_DIR_V2 = 'path/to/Dataset_v2'
#
# Khuyến nghị: Mode B — tự tay chia 80% real → train, 20% real → val.
#   Điều này đảm bảo tập val có data thực tế → accuracy hiển thị
#   phản ánh đúng độ chính xác ngoài thực địa.
# ============================================================

# ── CẤU HÌNH DATA ────────────────────────────────────────────
DATA_MODE   = 'flat'           # 'flat' hoặc 'split'
DATA_DIR_V2 = _DEFAULT_DATA    # ← ĐỔI PATH NÀY thành Dataset_v2 của bạn
                                # VD: './Dataset_v2' hoặc '../DataSet_v2'

# Chỉ dùng khi DATA_MODE = 'split'
TRAIN_SUBDIR = 'train'
VAL_SUBDIR   = 'val'

# Val split ratio — chỉ dùng khi DATA_MODE = 'flat'
VAL_RATIO = 0.2
# ────────────────────────────────────────────────────────────


def load_samples_from_dir(data_dir, desc=''):
    """Load tất cả samples từ thư mục cấu trúc class/images."""
    data_path = _resolve_data_root(data_dir)
    print(f'[INFO] {desc} root: {data_path}')

    folder_map       = {f.name: f for f in data_path.iterdir() if f.is_dir()}
    canonical_to_dir = {}
    for fn, fp in folder_map.items():
        cn = _canonical_from_folder(fn)
        if cn and cn not in canonical_to_dir:
            canonical_to_dir[cn] = fp
    for cls_name in CLASSES:
        exact = data_path / cls_name
        if exact.exists() and exact.is_dir():
            canonical_to_dir[cls_name] = exact

    missing = [c for c in CLASSES if c not in canonical_to_dir]
    if missing:
        print(f'[WARN] Missing class folders: {missing} — sẽ bỏ qua')

    samples = []
    counts  = {}
    for cls_name in CLASSES:
        if cls_name not in canonical_to_dir:
            counts[cls_name] = 0
            continue
        cls_dir = canonical_to_dir[cls_name]
        cls_idx = CLASS2IDX[cls_name]
        cnt = 0
        for ext in ('*.jpg','*.jpeg','*.png','*.bmp','*.webp',
                    '*.JPG','*.JPEG','*.PNG','*.BMP','*.WEBP'):
            for p in cls_dir.glob(ext):
                samples.append((str(p), cls_idx))
                cnt += 1
        counts[cls_name] = cnt

    return samples, counts


# ── Load samples theo mode ────────────────────────────────────
if DATA_MODE == 'split':
    train_dir = str(Path(DATA_DIR_V2) / TRAIN_SUBDIR)
    val_dir   = str(Path(DATA_DIR_V2) / VAL_SUBDIR)
    trn_samples, trn_counts = load_samples_from_dir(train_dir, 'TRAIN')
    val_samples, val_counts = load_samples_from_dir(val_dir,   'VAL')
    print(f'\n  Train: {len(trn_samples)} | Val: {len(val_samples)}')

elif DATA_MODE == 'flat':
    all_samples, all_counts = load_samples_from_dir(DATA_DIR_V2, 'ALL')
    trn_samples, val_samples = train_val_split_stratified(
        all_samples, val_ratio=VAL_RATIO, seed=CONFIG['seed'])
    trn_counts = {}; val_counts = {}
    for _, lbl in trn_samples:
        cls = IDX2CLASS[lbl]; trn_counts[cls] = trn_counts.get(cls, 0) + 1
    for _, lbl in val_samples:
        cls = IDX2CLASS[lbl]; val_counts[cls] = val_counts.get(cls, 0) + 1
    print(f'\n  Auto-split (val={VAL_RATIO:.0%}): '
          f'Train={len(trn_samples)} | Val={len(val_samples)}')

else:
    raise ValueError(f'DATA_MODE phải là flat hoặc split, nhận: {DATA_MODE!r}')

# ── Phân tích phân phối ────────────────────────────────────────
print(f'\n  {"Class":<20} {"Train":>8} {"Val":>8} {"Total":>8}')
print('  ' + '-'*48)
for cls in CLASSES:
    t = trn_counts.get(cls, 0)
    v = val_counts.get(cls, 0)
    print(f'  {cls:<20} {t:>8} {v:>8} {t+v:>8}')

total_t = sum(trn_counts.values())
total_v = sum(val_counts.values())
print(f'  {"TOTAL":<20} {total_t:>8} {total_v:>8} {total_t+total_v:>8}')

trn_arr = np.array([trn_counts.get(c, 0) for c in CLASSES], dtype=float)
if trn_arr.min() < 10:
    low = [CLASSES[i] for i in np.where(trn_arr < 10)[0]]
    print(f'\n  [WARN] Classes có < 10 ảnh train: {low}')
    print('         Cân nhắc thu thập thêm dữ liệu cho các class này.')
ratio = trn_arr.max() / max(trn_arr.min(), 1)
print(f'\n  Imbalance ratio: {ratio:.1f}x '
      f'(WeightedRandomSampler sẽ xử lý tự động)')

print('\n[OK] Data preparation done.')


In [ ]:
# ============================================================
# CELL 8 - PHASE 6: REAL-WORLD FINE-TUNING
# ============================================================
# Nguyên tắc chống Catastrophic Forgetting:
#   - LR rất nhỏ: backbone 1e-6, head 3e-5
#   - Differential LR: backbone < head (bảo vệ features đã học)
#   - Augmentation 'lighting': RandomResizedCrop + AGC + Shadow + HardLight
#   - EMA decay cao (0.9998): checkpoint trung bình ổn định hơn
#   - Early stop patience=8 epochs
# ============================================================

def phase6_finetune(trn_samples, val_samples, config=CONFIG,
                    base_ckpt=None, extra_epochs=None):
    device   = torch.device(config['device'])
    epochs   = extra_epochs or config.get('p6_epochs', 15)
    img_size = config.get('p6_img_size', 384)

    print(f"\n{'='*70}")
    print(f"  PHASE 6: REAL-WORLD FINE-TUNING  |  Device: {device}")
    print(f"  Epochs: {epochs}  |  Batch: {config.get('p6_batch', 8)}")
    print(f"  LR: head={config['p6_lr_head']:.0e}  "
          f"backbone={config['p6_lr_backbone']:.0e}")
    print(f"  Augmentation: {config.get('p6_aug', 'lighting')}")
    print(f"{'='*70}\n")

    # ── Load checkpoint ────────────────────────────────────────────────────
    ckpt_to_load = base_ckpt or config.get('p6_base_ckpt', 'best_model.pth')
    out_dir      = config.get('output_dir', './outputs')
    ckpt_p6      = os.path.join(out_dir, 'best_model_p6.pth')
    os.makedirs(out_dir, exist_ok=True)

    model = WasteDetector(NUM_CLASSES, pretrained=False).to(device)

    if os.path.exists(ckpt_to_load):
        ck = torch.load(ckpt_to_load, map_location=device, weights_only=False)
        state = ck.get('model_state', ck.get('ema_state', ck))
        model.load_state_dict(state, strict=True)
        base_best = float(ck.get('val_acc', 0.0))
        src       = ck.get('source', 'unknown')
        print(f"  [OK] Loaded: {ckpt_to_load}")
        print(f"       val_acc={base_best:.4f}  source={src}")
    else:
        raise FileNotFoundError(
            f"Checkpoint không tìm thấy: {ckpt_to_load}\n"
            f"Đặt CONFIG['p6_base_ckpt'] = path đến file .pth của Phase 5."
        )

    # ── Dataloader ─────────────────────────────────────────────────────────
    batch_size = config.get('p6_batch', _B['p3'] * _SCALE)
    trn_dl, val_dl = make_dataloaders(
        trn_samples, val_samples, img_size, batch_size,
        config.get('p6_aug', 'lighting'),
        config['num_workers'], device.type,
        config=config
    )

    # ── Unfreeze toàn bộ, differential LR ─────────────────────────────────
    # Không lock bất kỳ layer nào, nhưng LR backbone << LR head
    # để bảo vệ các đặc trưng thấp cấp (edges, textures) đã học tốt.
    for p in model.parameters():
        p.requires_grad = True

    focal_alpha = compute_class_alpha(trn_samples, power=config.get('alpha_power', 0.7))
    criterion   = FocalLoss(
        config['focal_gamma'], focal_alpha,
        label_smoothing=config.get('p6_label_smooth', 0.02)
    ).to(device)

    param_groups = make_param_groups(
        model,
        lr_head=config['p6_lr_head'],
        lr_bb=config['p6_lr_backbone']
    )
    optimizer = optim.AdamW(param_groups, weight_decay=config['weight_decay'])

    # CosineAnnealingLR: LR giảm từ p6_lr_head về 1e-6 theo hình cos
    # Tránh "catastrophic update" ở epoch cuối nếu LR không giảm
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )

    # EMA decay cao hơn → model trung bình "nhớ" lâu hơn → ổn định hơn
    ema    = ModelEMA(model, decay=config.get('p6_ema_decay', 0.9998))
    scaler = (torch.amp.GradScaler(enabled=True) if device.type == 'cuda'
              else torch.amp.GradScaler('cpu', enabled=False))

    # ── Training state ─────────────────────────────────────────────────────
    best_acc   = base_best
    best_w     = copy.deepcopy(model.state_dict())
    no_improve = 0
    patience   = config.get('p6_patience', 8)

    history = {
        'train_loss': [], 'train_acc': [], 'val_loss': [],
        'val_acc': [], 'ema_val_acc': [], 'lr': [], 'phase': []
    }

    print(f'  Train: {len(trn_samples)} | Val: {len(val_samples)}')
    print(f'  Base accuracy: {base_best:.4f}')
    print(f'  Early stop patience: {patience} epochs\n')

    for ep in range(1, epochs + 1):
        t0 = time.time()

        tl, ta = train_one_epoch(
            model, trn_dl, optimizer, criterion, device,
            mixup=config.get('p6_mixup', 0.1),
            cutmix=config.get('p6_cutmix', 0.0),
            scaler=scaler, epoch=ep
        )
        ema.update(model, ep)

        vl, va, per_cls = validate(model, val_dl, criterion, device)
        _, ea, _        = validate(ema.ema, val_dl, criterion, device)

        scheduler.step()
        lr = optimizer.param_groups[-1]['lr']

        eff_acc = max(va, ea)
        eff_src = 'EMA' if ea > va else 'model'
        dt      = time.time() - t0

        print(f"  P6[{ep:02d}/{epochs}] "
              f"L:{tl:.3f} A:{ta:.3f} | "
              f"vL:{vl:.3f} vA:{va:.3f} eA:{ea:.3f} | "
              f"lr:{lr:.1e} {dt:.0f}s{gpu_mem_str()}")

        if ep % 3 == 0 or ep == epochs:
            print('  Per-cls: ' +
                  ' | '.join(f'{k[:8]}:{v:.2f}' for k, v in per_cls.items()))

        _track(history, tl, ta, vl, va, ea, 6, lr)

        if eff_acc > best_acc:
            best_acc = eff_acc
            best_w   = copy.deepcopy(
                ema.ema.state_dict() if ea > va else model.state_dict()
            )
            torch.save({
                'epoch'         : ep,
                'model_state'   : best_w,
                'val_acc'       : best_acc,
                'classes'       : CLASSES,
                'img_size'      : img_size,
                'agc_target'    : config['agc_target'],
                'agc_gamma_min' : config['agc_gamma_min'],
                'agc_gamma_max' : config['agc_gamma_max'],
                'source'        : f'phase6_realworld_{eff_src}',
                'base_acc'      : base_best,
            }, ckpt_p6)
            delta = (best_acc - base_best) * 100
            print(f'  >> Saved P6 best: {best_acc:.4f} ({eff_src})  '
                  f'[{"+" if delta>=0 else ""}{delta:.2f}% vs base]')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'\n  >> Early stop ep{ep} '
                      f'(no improve {no_improve}/{patience})')
                break

        cleanup_gpu()

    # ── Finalize ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  Phase 6 DONE")
    print(f"  Base acc    : {base_best:.4f}")
    print(f"  Best P6 acc : {best_acc:.4f}  "
          f"({'↑' if best_acc > base_best else '↓'}"
          f"{abs(best_acc - base_best)*100:.2f}%)")
    print(f"  Checkpoint  : {ckpt_p6}")

    model.load_state_dict(best_w)

    print('\n  Final evaluation...')
    _, _, _, ap, al = validate(model, val_dl, criterion, device, return_preds=True)
    plot_confusion_matrix(ap, al, out_dir)
    plot_history(history, out_dir)

    if best_acc <= base_best:
        print('\n  [WARN] Phase 6 KHÔNG cải thiện. Kiểm tra:')
        print('    1. DataRealTest đủ ảnh? (>30 ảnh/class là tối thiểu)')
        print('    2. Label có bị sai? (xem confusion matrix)')
        print('    3. Thử tăng epochs hoặc thu thập thêm data.')

    del trn_dl, val_dl
    cleanup_gpu()
    return model, best_acc


# ── RUN ──────────────────────────────────────────────────────────────────────
seed_everything(CONFIG['seed'])
p6_model, p6_acc = phase6_finetune(trn_samples, val_samples, config=CONFIG)
print(f'\nPhase 6 best val acc: {p6_acc:.4f}')


In [ ]:
# ============================================================
# CELL 9 - EXPORT ONNX v2  (Phase 6 checkpoint)
# ============================================================
%pip install onnxscript -q

def export_onnx_v2(model, config=CONFIG, img_size=None, suffix='_v2'):
    """Export Phase 6 model → ONNX + model_meta.json."""
    img_size  = img_size or config.get('p6_img_size', 384)
    out_dir   = config.get('output_dir', './outputs')
    os.makedirs(out_dir, exist_ok=True)

    cpu_model = copy.deepcopy(model).cpu().eval()
    dummy     = torch.randn(1, 3, img_size, img_size)
    onnx_name = f'waste7_detector{suffix}.onnx'
    out_path  = os.path.join(out_dir, onnx_name)

    torch.onnx.export(
        cpu_model, (dummy,), out_path,
        input_names=['image'], output_names=['logits', 'objectness'],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes={
            'image'      : {0: 'batch'},
            'logits'     : {0: 'batch'},
            'objectness' : {0: 'batch'},
        },
    )
    del cpu_model, dummy; gc.collect()
    print(f'  >> ONNX v2  : {out_path}')

    # model_meta.json — testPC.py đọc file này để cấu hình AGC
    meta = {
        'classes'       : CLASSES,
        'img_size'      : img_size,
        'agc_target'    : config['agc_target'],
        'agc_gamma_min' : config['agc_gamma_min'],
        'agc_gamma_max' : config['agc_gamma_max'],
        'phase'         : 'phase6_realworld',
    }
    meta_path = os.path.join(out_dir, 'model_meta.json')
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'  >> Meta     : {meta_path}')
    print(f'     AGC: target={config["agc_target"]}  '
          f'clip=[{config["agc_gamma_min"]}, {config["agc_gamma_max"]}]')

    print(f'\n  Deploy:')
    print(f'    1. Copy {onnx_name} + model_meta.json → thư mục testPC.py')
    print(f'    2. Đổi ONNX_PATH = "waste7_detector{suffix}.onnx" trong testPC.py')
    return out_path


# Chạy export từ model Phase 6
export_onnx_v2(p6_model, CONFIG)
